<a href="https://colab.research.google.com/github/Sathyaseelan-Geo-Spatial-Lab/AQI_App/blob/main/code/python/PM2_5_AQ_Online_2026_CaseStudy_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Background

This is a code notebook created for the "[Estimating Surface PM<sub>2.5</sub> Using Satellite Data and Other Information Sources](https://www.earthdata.nasa.gov/learn/trainings/estimating-surface-pm2.5-using-satellite-data-other-information-sources)" [ARSET](https://www.earthdata.nasa.gov/data/projects/arset) training. The purpose is to illustrate how to extract data from existing satellite-derived PM<sub>2.5</sub> data products for a specified region of interest and compare these to ground-based PM<sub>2.5</sub> measurement data.

Authors:  
*   Carl Malings / Morgan State University & NASA Goddard Space Flight Center
*   Sebastián Diez / Centro de Investigación en Tecnologías para la Sociedad, Universidad del Desarrollo (Chile)

# Setup

These are the necessary setup steps to prepare for the case study:

1.   Copy this notebook to a place in your Google Drive.
  -   Select: File > Save a Copy in Drive
  -   I suggest making a new folder to store all the codes and data for this case study, to keep your Drive organized. I named my folder `PM25_Training_2026` and placed it within my Drive at `/content/drive/MyDrive/Colab Notebooks/ARSET/`.

2.   Create a folder in your Google Drive where the SatPM and MERRA2_CNN data will be uploaded. I named these folders `SatPM_Data` and `MERRA2_CNN_Data`, and placed them within the `PM25_Training_2026` folder.

3. Run the code block below to start up the Colab environment, load the required packages, and input your login information. Required login details are:
* [NASA Earthdata login and password](https://urs.earthdata.nasa.gov/).
* [OpenAQ API key](https://docs.openaq.org/using-the-api/api-key).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install & Import the required packages:
!pip install --quiet cartopy earthaccess

# basic packages:
import os
import getpass
from google.colab import drive
import warnings
warnings.filterwarnings('ignore') # ignore warnings

# data analysis packages:
import numpy as np
import pandas as pd
import xarray as xr

# data access packages:
import earthaccess
import requests
import urllib
import re
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# plotting packages:
import matplotlib.pyplot as plt
import cartopy
import seaborn as sns

# Connect to Google Dive:
drive.mount('/content/drive')

# Store login information:
earthaccess.login()
s_key_openAQ = getpass.getpass('Enter OpenAQ API key:')

## Define useful functions

In [ ]:
# Define useful functions:
def F_subset_and_combine(l_files,
                         n_lon_min,
                         n_lat_min,
                         n_lon_max,
                         n_lat_max,
                         t_start,
                         t_end,
                         s_lat= 'lat',
                         s_lon = 'lon',
                         s_time = 'time',
                         F_time_from_file = None,
                         ):
  '''
  This function takes of list of files, combines the data in these files together, and trims to latitude, longitude, and time of interest.
  Inputs:
    l_files: a list of files to combine.
    n_lon_min: minimum longitude of the region of interest.
    n_lon_max: maximum longitude of the region of interest.
    n_lat_min: minimum latitude of the region of interest.
    n_lat_max: maximum latitude of the region of interest.
    t_start: start time of interest (as numpy datetime64 type).
    t_end: end time of interest (as numpy datetime64 type).
    s_lat: name of the dimension for latitude.
    s_lon: name of the dimension for longitude.
    s_time: name of the dimension for time.
    F_time_from_file: function which take the file path as an input and returns a timestamp to assign to the data in the file. Used to fill in the time dimension in single-timestamp datasets.
  Outputs:
    a_data: an xarray object with the data from the region and time period of interest.
  '''
  l_data_loaded = []
  for s_file in l_files:
    a_data_file = xr.open_dataset(s_file)
    if s_time not in a_data_file.dims:
      a_data_file = a_data_file.expand_dims({s_time:[F_time_from_file(s_file)]})
    # a_data_file_subset = a_data_file.sel(lat=slice(n_lat_min,n_lat_max),lon=slice(n_lon_min,n_lon_max),time=slice(t_start,t_end))
    a_data_file_subset = a_data_file.loc[{s_lat:slice(n_lat_min,n_lat_max),s_lon:slice(n_lon_min,n_lon_max),s_time:slice(t_start,t_end)}]
    l_data_loaded += [a_data_file_subset]
  a_data = xr.concat(l_data_loaded, dim=s_time)
  return a_data

def F_get_OpenAQ_from_API(n_lon_min,
                         n_lat_min,
                         n_lon_max,
                         n_lat_max,
                         t_start,
                         t_end,
                         l_variables=['PM2.5'],
                         b_reference = True,
                         b_lowcost = False,
                         s_key=s_key_openAQ):
  '''
  This function queries OpenAQ for ground-based air quality monitor data in a specified region of interest and time period.
  Inputs:
    n_lon_min: minimum longitude of the region of interest.
    n_lon_max: maximum longitude of the region of interest.
    n_lat_min: minimum latitude of the region of interest.
    n_lat_max: maximum latitude of the region of interest
    t_start: start time of interest (as numpy datetime64 type).
    t_end: end time of interest (as numpy datetime64 type).
    l_variables: list of variables to query.
    b_reference: if True, reference monitors will be queried.
    b_lowcost: if True, low-cost monitors will be queried.
    s_key: OpenAQ API key.
  Outputs:
    f_data: a pandas dataframe with the data from the region and time period of interest.
  '''

  # Some dictionaries defining the data types and unit conversions:
  d_params = {'CO':'co',
              'O3':'o3',
              'NO2':'no2',
              'SO2':'so2',
              'PM2.5':'pm25',
              'PM10':'pm10',
              'BC':'bc',
              }
  d_unit_converstions = {'CO':{'ppm':1000,
                                'ppb':1,
                                'µg/m³':0.873},
                        'O3':{'ppm':1000,
                              'ppb':1,
                              'µg/m³':0.500},
                        'NO2':{'ppm':1000,
                              'ppb':1,
                              'µg/m³':0.532},
                        'SO2':{'ppm':1000,
                              'ppb':1,
                              'µg/m³':0.382},
                        'PM2.5':{'ppm':1275.4,
                                'ppb':1.2754,
                                'µg/m³':1},
                        'PM10':{'ppm':1275.4,
                                'ppb':1.2754,
                                'µg/m³':1},
                        'BC':{'ppm':1275.4,
                              'ppb':1.2754,
                              'µg/m³':1},
                        }

  # Start and End times as pandas datetime format:
  t_start = pd.to_datetime(t_start,utc=True)
  t_end = pd.to_datetime(t_end,utc=True)
  # Query for parameter IDs
  s_parameter_query_base = 'https://api.openaq.org/v3/parameters'
  response_parameters = requests.get(s_parameter_query_base,headers={'X-API-Key':s_key})
  if (response_parameters.status_code == 200):
    d_response_parameters = response_parameters.json()
    l_parameter_ids = []
    for d_response_parameter in d_response_parameters['results']:
      for s_variable in l_variables:
        if d_response_parameter['name'] == d_params[s_variable]:
          l_parameter_ids += [d_response_parameter['id']]
    s_parameter_ids = ','.join([str(n_parameter_id) for n_parameter_id in l_parameter_ids])
  else:
    print('ERROR: Parameter Query Failed')

  # Query for sensor IDs
  if (b_reference & (not b_lowcost)):
    s_location_query_base = 'https://api.openaq.org/v3/locations?limit=1000&page={n_page}&bbox={n_lon_min},{n_lat_min},{n_lon_max},{n_lat_max}&parameters_id={s_parameter_ids}&mobile=false&monitor=true'
  elif (~b_reference & b_lowcost):
    s_location_query_base = 'https://api.openaq.org/v3/locations?limit=1000&page={n_page}&bbox={n_lon_min},{n_lat_min},{n_lon_max},{n_lat_max}&parameters_id={s_parameter_ids}&mobile=false&monitor=false'
  else:
    s_location_query_base = 'https://api.openaq.org/v3/locations?limit=1000&page={n_page}&bbox={n_lon_min},{n_lat_min},{n_lon_max},{n_lat_max}&parameters_id={s_parameter_ids}&mobile=false'

  n_page = 1
  REQUERY = True
  l_sensor_ids = []
  d_sensor_latitude = {}
  d_sensor_longitude = {}
  while REQUERY:
    s_location_query = s_location_query_base.format(n_page=n_page,
                                                    n_lat_min=n_lat_min,
                                                    n_lon_min=n_lon_min,
                                                    n_lat_max=n_lat_max,
                                                    n_lon_max=n_lon_max,
                                                    s_parameter_ids=s_parameter_ids)
    response_locations = requests.get(s_location_query,headers={'X-API-Key':s_key})
    if (response_locations.status_code == 200):
      d_response_locations = response_locations.json()
      for d_location in d_response_locations['results']:
        try:
          if ((pd.to_datetime(d_location['datetimeFirst']['utc']) <= t_end) and (pd.to_datetime(d_location['datetimeLast']['utc']) >= t_start)):
            for d_sensor in d_location['sensors']:
              if d_sensor['parameter']['id'] in l_parameter_ids:
                l_sensor_ids += [d_sensor['id']]
                d_sensor_latitude[d_sensor['id']] = d_location['coordinates']['latitude']
                d_sensor_longitude[d_sensor['id']] = d_location['coordinates']['longitude']
        except:
          pass
      if d_response_locations['meta']['found'] >= (n_page*d_response_locations['meta']['limit']):
        n_page += 1
      else:
        REQUERY = False
    else:
      REQUERY = False
      print('ERROR: Locations Query Failed')

  # Query for Data:
  s_data_query_base = 'https://api.openaq.org/v3/sensors/{n_sensor_id}/measurements?limit=1000&page={n_page}&datetime_from={s_datetime_from}&datetime_to={s_datetime_to}'

  l_data = []
  for n_sensor_id in l_sensor_ids:
    n_page = 1
    REQUERY = True
    while REQUERY:
      s_data_query = s_data_query_base.format(n_sensor_id=n_sensor_id,
                                              n_page=n_page,
                                              s_datetime_from = str(t_start)[0:10]+'T'+str(t_start)[11:19]+'Z',
                                              s_datetime_to = str(t_end)[0:10]+'T'+str(t_end)[11:19]+'Z'
                                              )
      response_data = requests.get(s_data_query,headers={'X-API-Key':s_key})
      if (response_data.status_code == 200):
        d_response_data = response_data.json()
        for i_datapoint,d_datapoint in enumerate(d_response_data['results']):
          if d_datapoint['value'] is not None:
            # Find the type of the data:
            s_type = [s_type for s_type in d_params.keys() if d_params[s_type] == d_datapoint['parameter']['name']][0]
            # Prepare to convert units:
            n_multiplier = d_unit_converstions[s_type][d_datapoint['parameter']['units']]
            n_value = d_datapoint['value']
            # Remove negative values:
            if n_value < 0:
              n_value = np.nan
            t_datapoint_start = pd.to_datetime(d_datapoint['period']['datetimeFrom']['utc'][0:19],utc=True)
            t_datapoint_end = pd.to_datetime(d_datapoint['period']['datetimeTo']['utc'][0:19],utc=True)
            d_datapoint_line = {'lat':d_sensor_latitude[n_sensor_id],
                                'lon':d_sensor_longitude[n_sensor_id],
                                'time':[(t_datapoint_start + (t_datapoint_end - t_datapoint_start)/2)],
                                s_type:[n_value*n_multiplier]}
            l_data += [pd.DataFrame.from_dict(d_datapoint_line)]
        print('Query completed: Sensor {}, page {}.'.format(n_sensor_id,n_page))
        if len(d_response_data['results']) == d_response_data['meta']['limit']:
          n_page += 1
        else:
          REQUERY = False
      else:
        REQUERY = False
        print('ERROR: Sensor {} Data Query Failed'.format(n_sensor_id))

  if len(l_data) > 0:
    f_data = pd.concat(l_data, ignore_index=True)
    # Correct for non-unique indeces
    f_data = f_data.groupby(['lat','lon','time']).mean()
  else:
    f_data = None

  return f_data

def F_plot_map(a_data_plot=None,
               a_data_plot_point=None,
               n_lat_min=-90,
               n_lat_max=90,
               n_lon_min=-180,
               n_lon_max=180,
               my_cmap = plt.get_cmap('Oranges'),
               Colorbar_Label = None,
               Plot_Title = None,
               Colorbar_Limits = [0,1],
               n_pointsize = 20,
               n_latlon_gridspacing = 1,
               o_figure = None,
               o_axis = None,
               o_plot = None,
               SHOW=True,
               b_shapefiles=True):
  '''
  This function creates a map plot from gridded or point data.
  Inputs:
    a_data_plot: gridded data (in xarray format) to plot on the map.
    a_data_plot_point: point data (in pandas table format) to plot on the map.
    n_lat_min: minimum latitude of the map.
    n_lat_max: maximum latitude of the map.
    n_lon_min: minimum longitude of the map.
    n_lon_max: maximum longitude of the map.
    my_cmap: color scale to be used by the plot.
    Colorbar_Label: label for the color scale.
    Plot_Title: title for the plot.
    Colorbar_Limits: limits for the color scale.
    n_pointsize: point size for the point data.
    n_latlon_gridspacing: spacing between latitude and longitude gridlines on the map
    o_figure: figure object to use for the plot.
    o_axis: axis object to use for the plot.
    o_plot: plot object to use for the plot.
    SHOW: if True, the plot will be shown.
    b_shapefiles: if True, borders and coastlines will be included in the plot.
  Outputs:
    o_figure: figure object used for the plot.
    o_axis: axis object used for the plot.
    o_plot: plot object used for the plot.
  '''
  if o_figure is None:
    o_figure,o_axis = plt.subplots(1,1)
    o_axis.remove()
    o_axis = o_figure.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
  if b_shapefiles:
    o_axis.coastlines()
    o_axis.add_feature(cartopy.feature.LAKES, edgecolor='black', facecolor='none', linestyle='-')
    o_axis.add_feature(cartopy.feature.BORDERS, edgecolor='black', facecolor='none', linestyle='--')
    o_axis.add_feature(cartopy.feature.STATES, edgecolor='black', facecolor='none', linestyle=':')
  o_axis.set_extent([np.max([n_lon_min-0.001,-180]), np.min([n_lon_max+0.001,180]), np.max([n_lat_min-0.001,-90]), np.min([n_lat_max+0.001,90])])
  gridlines = o_axis.gridlines(crs=cartopy.crs.PlateCarree(), xlocs=range(np.int_(np.floor(np.max([n_lon_min,-180]))),np.int_(np.ceil(np.min([n_lon_max,180])))+1,n_latlon_gridspacing), ylocs=range(np.int_(np.floor(np.max([n_lat_min,-90]))),np.int_(np.ceil(np.min([n_lat_max,90])))+1,n_latlon_gridspacing), draw_labels=True, linewidth=1, color='black', alpha=0.5, linestyle='--')
  gridlines.top_labels = False
  gridlines.right_labels = False
  if a_data_plot is not None:
    if 'time' in a_data_plot.dims:
      # take average for plotting:
      a_data_plot = a_data_plot.mean(dim='time')
  else:
    # Create a synthetic dataset to enable a colorbar:
    if Colorbar_Limits is None:
      a_data_plot = xr.DataArray([[Colorbar_Limits[0],Colorbar_Limits[0]],[Colorbar_Limits[1],Colorbar_Limits[1]]],dims=('lat','lon'),coords={'lat':[n_lat_max+1,n_lat_max+2],'lon':[n_lon_max+1,n_lon_max+2]})
    else:
      a_data_plot = xr.DataArray([[0,1],[2,3]],dims=('lat','lon'),coords={'lat':[n_lat_max+1,n_lat_max+2],'lon':[n_lon_max+1,n_lon_max+2]})
  if Colorbar_Limits is None:
    if Colorbar_Label is not None:
      o_plot = a_data_plot.plot(ax=o_axis,cmap=my_cmap,cbar_kwargs={'label': Colorbar_Label})
    else:
      o_plot = a_data_plot.plot(ax=o_axis,cmap=my_cmap)
  else:
    if Colorbar_Label is not None:
      o_plot = a_data_plot.plot(ax=o_axis,cmap=my_cmap,vmin=Colorbar_Limits[0],vmax=Colorbar_Limits[1],cbar_kwargs={'label': Colorbar_Label})
    else:
      o_plot = a_data_plot.plot(ax=o_axis,cmap=my_cmap,vmin=Colorbar_Limits[0],vmax=Colorbar_Limits[1])
  if a_data_plot_point is not None:
    if type(a_data_plot_point) is pd.Series:
        f_data_plot_point = a_data_plot_point
        if 'time' in f_data_plot_point.index.names:
            f_data_plot_point = f_data_plot_point.groupby(['lat','lon']).mean('time')
        v_lat_plot = f_data_plot_point.reset_index()['lat'].values
        v_lon_plot = f_data_plot_point.reset_index()['lon'].values
        v_data_temp = f_data_plot_point.values
    else:
        v_lat_plot = np.array(a_data_plot_point.lat)
        v_lon_plot = np.array(a_data_plot_point.lon)
        if 'time' in a_data_plot_point.dims:
          # take average for plotting:
          v_data_temp = np.array(a_data_plot_point.mean(dim='time'))
        else:
          v_data_temp = np.array(a_data_plot_point)
    if Colorbar_Limits is None:
      o_plot_point = o_axis.scatter(x=v_lon_plot,y=v_lat_plot,s=n_pointsize,edgecolors='k',c=v_data_temp,cmap=my_cmap)
    else:
      o_plot_point = o_axis.scatter(x=v_lon_plot,y=v_lat_plot,s=n_pointsize,edgecolors='k',c=v_data_temp,vmin=Colorbar_Limits[0],vmax=Colorbar_Limits[1],cmap=my_cmap)
  if Plot_Title is not None:
    o_axis.set_title(Plot_Title)
  if SHOW:
    o_figure.show()
  return o_figure,o_axis,o_plot

def F_compute_metrics(f_data,
                      s_time = 'time_month_start',
                      s_value = 'Average',
                      s_estimate = 'SatPM',
                      b_print = True):

  '''
  This function computes standard performance metrics for a dataset.
  Inputs:
    f_data: a pandas dataframe with the data. The index needs to have three levels, corresonding to latitude, longitude, and time.
    s_time: name of the dimension for time.
    s_value: name of the column containing the measured values.
    s_estimate: name of the column containing the estimated values.
    b_print: if True, the metrics will be printed.
  Outputs:
    d_metrics: a dictionary of metric values computed from the data.
  '''
  d_metrics = {}

  f_data['error'] = f_data[s_estimate] - f_data[s_value]
  d_metrics['Estimated Mean'] = f_data[s_estimate].mean()
  d_metrics['Measured Mean'] = f_data[s_value].mean()
  d_metrics['Bias'] = f_data['error'].mean()
  d_metrics['Mean Normalized Bias'] = d_metrics['Bias']/d_metrics['Measured Mean']
  d_metrics['RMSE'] = np.sqrt((f_data['error']**2).mean())
  d_metrics['Mean Normalized RMSE'] = d_metrics['RMSE']/d_metrics['Measured Mean']
  d_metrics['R^2'] = 1 - (f_data['error']**2).sum()/((f_data[s_value] - d_metrics['Measured Mean'])**2).sum()

  unique_coordinates = f_data.index.droplevel(s_time).unique()
  unique_times = f_data.index.unique(level=s_time)

  # spatial correlation
  if len(unique_coordinates) > 1:
    l_spatial_correlation_by_time = []
    for t_time in unique_times:
      f_data_temp = f_data.xs(t_time,level=s_time)
      try:
        l_spatial_correlation_by_time += [f_data_temp[s_value].corr(f_data_temp[s_estimate])]
      except:
        pass
    d_metrics['Spatial Correlation - median'] = np.nanmedian(l_spatial_correlation_by_time)
    d_metrics['Spatial Correlation - maximum'] = np.nanmax(l_spatial_correlation_by_time)
    d_metrics['Spatial Correlation - minimum'] = np.nanmin(l_spatial_correlation_by_time)

  # temporal correlation
  if len(unique_times) > 1:
    l_temporal_correlation_by_grid = []
    for coord in unique_coordinates:
      f_data_temp = f_data.loc[coord]
      try:
        l_temporal_correlation_by_grid += [f_data_temp[s_value].corr(f_data_temp[s_estimate])]
      except:
        pass
    d_metrics['Temporal Correlation - median'] = np.nanmedian(l_temporal_correlation_by_grid)
    d_metrics['Temporal Correlation - maximum'] = np.nanmax(l_temporal_correlation_by_grid)
    d_metrics['Temporal Correlation - minimum'] = np.nanmin(l_temporal_correlation_by_grid)

  if b_print:
    for s_metric in d_metrics:
      if ' - ' in  s_metric:
        s_metric_base = s_metric.split(' - ')[0]
        s_metric_modifier = s_metric.split(' - ')[1]
        if s_metric_modifier == 'median':
          print(f'{s_metric_base}: {d_metrics[s_metric]:.3g} ({d_metrics[s_metric_base+' - minimum']:.3g} to {d_metrics[s_metric_base+' - maximum']:.3g})')
      else:
        print(f'{s_metric}: {d_metrics[s_metric]:.3g}')

  return d_metrics

## Define SINCA data scraping functions (Chile ground-truth source)

These functions scrape ground-based PM<sub>2.5</sub> monitor data from Chile's official monitoring network **SINCA**
(Sistema de Informacion Nacional de Calidad del Aire).
They are used as an alternative to OpenAQ for this case study.

**Why SINCA instead of OpenAQ for this case study?** OpenAQ ingests mostly
*validated* records. For recent or extreme episodes (e.g., the February 2023 megafires) Chilean stations report data flagged as *not validated* in SINCA, so those values are largely absent from OpenAQ - precisely when the satellite signal is strongest.
OpenAQ also collects PM<sub>2.5</sub> monitor data from SINCA as 24-hour rolling averages. We'd like to look at the hourly-average data in this case study, to better capture the extremes due to the fires.
SINCA exposes validated, preliminary and non-validated series of hourly-average data, so it captures the fire episode.
The adapter below returns data in the same format as `F_get_OpenAQ_from_API`, so the rest of the notebook is unchanged.

In this case study, we'll use both OpenAQ and the SINCA data scraper, and compare the results, to illustrate why we chose to use the SINCA data obtained from the data scraper instead of from OpenAQ. **In general, it is always best to check the original source of the data; the local researchers and experts who have collected the data will know best about how to use the data appropriately.**

In [ ]:
# Endpoints of the SINCA public site:
BASE        = "https://sinca.mma.gob.cl"
REGION_URL  = f"{BASE}/index.php/region/index/id"
TS_CGI      = f"{BASE}/cgi-bin/APUB-MMA/apub.tsindico2.cgi"
HEADERS     = {"User-Agent": "Mozilla/5.0"}

def yymmdd(date_like):
    """Convert a date like '2023-02-01' to SINCA's YYMMDD format."""
    return pd.to_datetime(date_like).strftime("%y%m%d")

def as_list(x):
    return list(x) if isinstance(x, (list, tuple, set)) else [x]

def get_sinca_region_stations(region_id):
    """List stations of a SINCA region. Example: region_id='VIII' (Biobio)."""
    r = requests.get(f"{REGION_URL}/{region_id}", headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    stations = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/index.php/estacion/index/id/" not in href:
            continue
        full_url = urljoin(BASE, href)
        m = re.search(r"/id/(\d+)", full_url)
        name = a.get_text(" ", strip=True)
        if m and name:
            stations.append({"region_id": region_id, "station_id": m.group(1),
                             "station_name": name, "station_url": full_url})
    return pd.DataFrame(stations).drop_duplicates().reset_index(drop=True)

def get_sinca_station_metadata(station_url):
    """Extract lat/lon (from the Google Maps JS) and basic metadata."""
    r = requests.get(station_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    html = r.text
    lat = lon = None
    m = re.search(r"new\s+google\.maps\.LatLng\(\s*([-+]?\d+\.\d+)\s*,\s*([-+]?\d+\.\d+)\s*\)", html)
    if m:
        lat, lon = float(m.group(1)), float(m.group(2))
    return {"lat": lat, "lon": lon}

def get_sinca_station_graph_links(station_url):
    """Find the data/graph links on a station page."""
    r = requests.get(station_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    rows = []
    for a in soup.find_all("a", href=True):
        href = urljoin(BASE, a["href"])
        if "apub.htmlindico2.cgi" in href:
            rows.append({"link_text": a.get_text(" ", strip=True), "graph_url": href})
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

def get_sinca_macro_options(graph_url):
    """Extract the 'macro' value (needed by the download CGI) from the left frame."""
    r = requests.get(graph_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    m_tag = re.search(r'<frame[^>]*\bid=["\']left["\'][^>]*>', r.text, flags=re.IGNORECASE | re.DOTALL)
    if not m_tag:
        raise RuntimeError("Left frame not found.")
    m_src = re.search(r'\bsrc="([^"]*)"', m_tag.group(0), flags=re.IGNORECASE)
    if not m_src:
        raise RuntimeError("Left frame src not found.")
    left_url = urljoin(BASE, m_src.group(1).replace("&amp;", "&"))
    r2 = requests.get(left_url, headers=HEADERS, timeout=30)
    r2.raise_for_status()
    soup = BeautifulSoup(r2.text, "html.parser")
    opts = [{"option_text": o.get_text(" ", strip=True), "macro_value": o.get("value")}
            for o in soup.select("select#ic option")]
    return pd.DataFrame(opts)

def _parse_sinca_hourly(raw_text):
    """Parse SINCA hourly CSV by COLUMN POSITION.
    Layout observed:  FECHA;HORA;<validated>;<preliminary>;<not_validated>;
    The first two columns are often empty; recent/extreme data lives in the
    'not_validated' column. We prefer validated > preliminary > not_validated,
    and keep the status so it can be reported honestly."""
    recs = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or line.startswith("FECHA"):
            continue
        p = line.split(";")
        if len(p) < 5:
            continue
        fecha, hora = p[0].strip(), p[1].strip()
        cols = [c.strip().replace(",", ".") for c in p[2:5]]
        def num(x):
            try: return float(x) if x != "" else None
            except ValueError: return None
        validated, preliminary, not_valid = num(cols[0]), num(cols[1]), num(cols[2])
        if validated is not None:
            val, status = validated, "validated"
        elif preliminary is not None:
            val, status = preliminary, "preliminary"
        elif not_valid is not None:
            val, status = not_valid, "not_validated"
        else:
            val, status = None, "no_data"
        recs.append({"datetime_local": pd.to_datetime("20" + fecha + hora,
                                                       format="%Y%m%d%H%M", errors="coerce"),
                     "PM2.5": val, "status": status})
    return pd.DataFrame(recs)

def _download_one_station_pm25(station_row, date_from, date_to):
    """Download hourly PM2.5 for a single SINCA station; None if unavailable."""
    df_links = get_sinca_station_graph_links(station_row["station_url"])
    cand = df_links[df_links["graph_url"].str.contains("PM25", case=False, na=False) &
                    df_links["graph_url"].str.contains("horario.horario", case=False, na=False)]
    if cand.empty:
        return None
    df_opt = get_sinca_macro_options(cand.iloc[0]["graph_url"])
    m = df_opt[df_opt["option_text"].str.contains("registro horario", case=False, na=False)]
    if m.empty:
        return None
    params = {"outtype": "xcl", "macro": m.iloc[0]["macro_value"],
              "from": yymmdd(date_from), "to": yymmdd(date_to),
              "path": "/usr/airviro/data/CONAMA/", "lang": "esp", "rsrc": "", "macropath": ""}
    r = requests.get(TS_CGI, params=params, headers=HEADERS, timeout=60)
    r.raise_for_status()
    raw = r.content.decode("utf-8", errors="replace")
    if "psgraph:" in raw or "Could not load macro" in raw:
        return None
    df = _parse_sinca_hourly(raw)
    meta = get_sinca_station_metadata(station_row["station_url"])
    df["lat"], df["lon"] = meta["lat"], meta["lon"]
    df["station_name"] = station_row["station_name"]
    return df

def F_get_SINCA_from_web(n_lon_min, n_lat_min, n_lon_max, n_lat_max,
                         t_start, t_end, regions, l_variables=["PM2.5"]):
    """Drop-in replacement for F_get_OpenAQ_from_API, reading from SINCA.

    Returns a pandas DataFrame with a 3-level (lat, lon, time) index and a
    'PM2.5' column, with time in UTC - exactly the format the rest of the
    notebook expects. SINCA serves whole regions, so pass the region ids that
    cover the bounding box; results are then clipped to the box.
    """
    frames = []
    for region_id in as_list(regions):
        try:
            df_st = get_sinca_region_stations(region_id)
        except Exception as e:
            print(f"REGION {region_id} listing failed: {e}")
            continue
        for _, row in df_st.iterrows():
            try:
                d = _download_one_station_pm25(row, pd.to_datetime(t_start), pd.to_datetime(t_end))
                if d is not None and len(d):
                    frames.append(d)
                    print(f"OK: [{region_id}] {row['station_name']} ({len(d)} rows)")
                else:
                    print(f"-- no hourly PM2.5: [{region_id}] {row['station_name']}")
            except Exception as e:
                print(f"FAIL: [{region_id}] {row['station_name']} -> {e}")
            time.sleep(0.2)
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    # SINCA reports in Chilean local time -> convert to UTC (to match OpenAQ / MERRA-2):
    df["time"] = (df["datetime_local"]
                  .dt.tz_localize("America/Santiago", ambiguous="NaT", nonexistent="NaT")
                  .dt.tz_convert("UTC"))
    # Clip to the bounding box (SINCA returns the whole region):
    df = df.dropna(subset=["lat", "lon", "time"])
    df = df[df["lat"].between(n_lat_min, n_lat_max) & df["lon"].between(n_lon_min, n_lon_max)]
    # Final desired format: (lat, lon, time) index + 'PM2.5' column:
    f_data = (df[["lat", "lon", "time", "PM2.5"]]
              .dropna(subset=["PM2.5"])
              .groupby(["lat", "lon", "time"]).mean())
    return f_data

## Define your domain of interest

1.   Use `n_lat_min`, `n_lat_max`, `n_lon_min`, and `n_lon_max` to define the minimum latitude, maximum latitude, minimum longitude, and maximum longitude of your domain of interest.


2.   Use `t_start` and `t_end` to specify a start and end time for your analysis interval. Time format is `'YYYY-MM-DD hh:mm:ss'`; time is UTC.

3.   Use `s_timezone` to specify the time zone in which the analysis is taking place. This will be useful for converting between UTC and local time. Check the [list of TZ database timezones](https://en.wikipedia.org/wiki/List_of_tz_database_time_zones) for possible options.

In [ ]:
# Define the bounding box:
n_lat_min = -38
n_lat_max = -32
n_lon_min = -73.75
n_lon_max = -69.75

# Define the start and end time:
t_start = np.datetime64('2022-12-01 00:00:00')
t_end = np.datetime64('2023-03-31 23:59:59')

# Specify the local time zone; this will be useful for later visualization:
s_timezome = 'America/Santiago'

# Let's look at the region we've selected to check out that it is what we want:
F_plot_map(n_lat_min=n_lat_min,
           n_lat_max=n_lat_max,
           n_lon_min=n_lon_min,
           n_lon_max=n_lon_max);

# Downloading Data

## Download SatPM Data

This code downloads data from the [SatPM V6.GL.03](https://www.satpm.org/v6-gl-03) dataset for our region and time period of interest.
Alternatively, you could manually download the files from the [SatPM data access website](https://www.satpm.org/data-access) and re-upload them to your Google Drive, but this code helps automate that process and avoids you needing to download the files to your computer.

In [ ]:
# Specify the folder in your drive where the data should be downloaded:
s_folder_SatPM = '/content/drive/MyDrive/Colab Notebooks/ARSET/PM25_Training_2026/SatPM_Data'

# Download the data for the time range of interest:
l_files_SatPM = []
for t_month in np.arange(t_start.astype('datetime64[M]'),t_end.astype('datetime64[M]')+np.timedelta64(1,'M'),np.timedelta64(1,'M')):
  s_year = str(t_month)[0:4]
  s_month = str(t_month)[5:7]
  s_url = f'https://s3.amazonaws.com/satpmdata/V6GL03/FineResolution/GL/Monthly/{s_year}/V6GL03.CNNPM25.GL.{s_year}{s_month}-{s_year}{s_month}.nc'
  s_file = s_url.split('/')[-1]
  s_download_path = os.path.join(s_folder_SatPM,s_file)
  if os.path.exists(s_download_path):
    print(f'{s_file} already downloaded.')
  else:
    urllib.request.urlretrieve(s_url,s_download_path)
    print(f'Downloaded {s_url} to {s_folder_SatPM}')
  l_files_SatPM += [s_download_path]

# Combine the data into one file:
def F_time_from_SatPM_url(s_url):
  s_YYYYMM = s_url.split('.')[-2].split('-')[0]
  return np.datetime64(f'{s_YYYYMM[0:4]}-{s_YYYYMM[4:6]}-01 00:00:00')
a_data_SatPM = F_subset_and_combine(l_files_SatPM,n_lon_min,n_lat_min,n_lon_max,n_lat_max,t_start,t_end,F_time_from_file = F_time_from_SatPM_url)

# Filter out negative values:
a_data_SatPM['PM25_filtered'] = a_data_SatPM['PM25'].where(a_data_SatPM['PM25'] > 0)

# Save the data so we can more easily come back to it in the future:
a_data_SatPM.to_netcdf(os.path.join(s_folder_SatPM,'SatPM_data_subset.nc'))

# Look at the Data Object:
a_data_SatPM

In [ ]:
# Plot the data:
F_plot_map(a_data_plot = a_data_SatPM['PM25_filtered'],
           n_lat_min=n_lat_min,
           n_lat_max=n_lat_max,
           n_lon_min=n_lon_min,
           n_lon_max=n_lon_max,
           Colorbar_Label = 'PM$_{2.5}$ [µg/m$^{3}$]',
           Plot_Title = f'SatPM\n(Average {t_start} to {t_end})',
           Colorbar_Limits=[0,40]);

## Download MERRA-2 CNN Data

This code downloads data from the [MERRA2_CNN_HAQAST_PM25](https://doi.org/10.5067/OCKK5HCFW5N3) dataset for our region and time period of interest.
This code uses the [earthaccess](https://earthaccess.readthedocs.io/en/stable/) python package to automate searching for and downloading the data.
Alternatively, you could manually download the files from using [Earthdata Search](https://search.earthdata.nasa.gov/search?q=MERRA2_CNN_HAQAST_PM25) and re-upload them to your Google Drive, but this code helps automate that process and avoids you needing to download the files to your computer.

In [ ]:
# Specify the folder in your drive where the data should be downloaded:
s_folder_MERRA2_CNN = '/content/drive/MyDrive/Colab Notebooks/ARSET/PM25_Training_2026/MERRA2_CNN_Data'

# Use earthaccess to search for the data:
l_results = earthaccess.search_data(
    short_name='MERRA2_CNN_HAQAST_PM25',
    bounding_box=(n_lon_min,n_lat_min,n_lon_max,n_lat_max),
    temporal=(str(t_start),str(t_end))
)

# Use earthaccess to download the data:
l_files_MERRA2_CNN = earthaccess.download(l_results,s_folder_MERRA2_CNN)

# Combine the data into one file:
a_data_MERRA2_CNN = F_subset_and_combine(l_files_MERRA2_CNN,n_lon_min,n_lat_min,n_lon_max,n_lat_max,t_start,t_end)

# Filter out values with low quality flags:
a_data_MERRA2_CNN['PM25_filtered'] = a_data_MERRA2_CNN['MERRA2_CNN_Surface_PM25'].where(a_data_MERRA2_CNN['QFLAG'] >= 3)

# Save the data so we can more easily come back to it in the future:
a_data_MERRA2_CNN.to_netcdf(os.path.join(s_folder_MERRA2_CNN,'MERRA2_CNN_data_subset.nc'))

# Look at the Data Object:
a_data_MERRA2_CNN

In [ ]:
# Plot the data:
F_plot_map(a_data_plot = a_data_MERRA2_CNN['PM25_filtered'],
           n_lat_min=n_lat_min,
           n_lat_max=n_lat_max,
           n_lon_min=n_lon_min,
           n_lon_max=n_lon_max,
           Colorbar_Label = 'PM$_{2.5}$ [µg/m$^{3}$]',
           Plot_Title = f'MERRA-2 CNN\n(Average {t_start} to {t_end})',
           Colorbar_Limits=[0,40]);

## Download Monitor Data from OpenAQ and the SINCA network

Typically, we might use a data aggregator like [OpenAQ API](https://docs.openaq.org/about/about) to search for and download ground-level monitoring data for our region and time period of interest. For this case study, we will also data directly from the Chilean national network **SINCA**, using the function written for this purpose, `F_get_SINCA_from_web`. We'll compare the two here, to illustrate why we ended up choosing the SINCA data for this case study.

In [ ]:
# Specify the folder in your drive where the data should be downloaded:
s_folder_Main = '/content/drive/MyDrive/Colab Notebooks/ARSET/PM25_Training_2026'

# Use OpenAQ to query the data:
f_data_monitors_OpenAQ = F_get_OpenAQ_from_API(n_lon_min,
                                               n_lat_min,
                                               n_lon_max,
                                               n_lat_max,
                                               t_start,
                                               t_end)

# Use a custom function to get data directly from SINCA:
f_data_monitors_SINCA = F_get_SINCA_from_web(n_lon_min,
                                             n_lat_min,
                                             n_lon_max,
                                             n_lat_max,
                                             t_start,
                                             t_end,
                                             regions=['V', 'M', 'VI', 'VII', 'XVI', 'VIII', 'IX'])
# SINCA region ids covering the bounding box (Roman numerals as used by SINCA):
# V=Valparaiso, M=Metropolitana, VI=O'Higgins, VII=Maule, XVI=Nuble, VIII=Biobio, IX=Araucania.

In [ ]:
# To compare the OpenAQ and SINCA datasets, we'll plot the data at a specific site.

# The "Parque O'Higgins" Monitor is at latitude -33.464142 longitude -70.660797
s_site = "Parque O'Higgins"
n_monitor_lat = -33.464142
n_monitor_lon = -70.660797

# Select just the data corresponding to this site:
f_data_monitors_OpenAQ_at_site = f_data_monitors_OpenAQ.loc[(n_monitor_lat,n_monitor_lon)]
f_data_monitors_SINCA_at_site = f_data_monitors_SINCA.loc[(n_monitor_lat,n_monitor_lon)]

# Plot the time series:
o_fig = plt.figure(figsize=(15,5))
o_ax = o_fig.add_subplot()
o_ax.plot(f_data_monitors_OpenAQ_at_site.index,f_data_monitors_OpenAQ_at_site['PM2.5'],color='blue',label='Data from OpenAQ')
o_ax.plot(f_data_monitors_SINCA_at_site.index,f_data_monitors_SINCA_at_site['PM2.5'],color='orange',label='Data from SINCA')
o_ax.legend();
o_ax.set_ylabel('PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title(f'Data comparison at {s_site}');


Comparing the time-series plots from OpenAQ and SINCA, it seems that the OpenAQ data are much smoother in time, and do not show some of the highest peak values associated with the February 2023 fires. In fact, the OpenAQ PM<sub>2.5</sub> data represent a 24-hour rolling values, not the hourly-average data reported by SINCA.

After consulting with local experts, we decided that the SINCA data would be more appropriate to the objectives of this case study.

In [ ]:
# Use the SINCA data as our monitor data:
f_data_monitors = f_data_monitors_SINCA

# Filter out values less than 0 and greater than 2000:
f_data_monitors['PM25_filtered'] = f_data_monitors['PM2.5'].where((f_data_monitors['PM2.5'] > 0) & (f_data_monitors['PM2.5'] < 2000))

# Save the data so we can more easily come back to it in the future:
f_data_monitors.to_csv(os.path.join(s_folder_Main,'Monitor_data_subset.csv'))

# Look at the Data Object:
f_data_monitors

In [ ]:
# Plot the data:
F_plot_map(a_data_plot_point = f_data_monitors['PM25_filtered'],
           n_lat_min=n_lat_min,
           n_lat_max=n_lat_max,
           n_lon_min=n_lon_min,
           n_lon_max=n_lon_max,
           Colorbar_Label = 'PM$_{2.5}$ [µg/m$^{3}$]',
           Plot_Title = f'Monitor Data\n(Average {t_start} to {t_end})',
           Colorbar_Limits=[0,40]);

# Analysis

Now, we will perform some analysis of the data we have accessed.








First, let's re-load our data from the data subset files saved to the Google Drives (this is useful if you are coming back to the analysis later, so you don't need to re-load the entire dataset).

In [ ]:
# Re-load the data from subsets saved in your Google Drive, if needed:
a_data_SatPM = xr.open_dataset(os.path.join(s_folder_SatPM,'SatPM_data_subset.nc'))
a_data_MERRA2_CNN = xr.open_dataset(os.path.join(s_folder_MERRA2_CNN,'MERRA2_CNN_data_subset.nc'))
f_data_monitors = pd.read_csv(os.path.join(s_folder_Main,'Monitor_data_subset.csv'),index_col=[0,1,2])

## Comparison using Maps

Let's make two comparison plots, overlaying the ground monitor data as points on top of the maps of the SatPM and MERRA-2 CNN datasets.

In [ ]:
# Plot: Compare Surface Montiors to SatPM
F_plot_map(a_data_plot = a_data_SatPM['PM25_filtered'],
           a_data_plot_point = f_data_monitors['PM25_filtered'],
           n_lat_min=n_lat_min,
           n_lat_max=n_lat_max,
           n_lon_min=n_lon_min,
           n_lon_max=n_lon_max,
           Colorbar_Label = 'PM$_{2.5}$ [µg/m$^{3}$]',
           Plot_Title = f'Comparing SatPM to Monitors\n(Average {t_start} to {t_end})',
           Colorbar_Limits=[0,50]);

# Plot: Compare Surface Montiors to MERRA-2 CNN
F_plot_map(a_data_plot = a_data_MERRA2_CNN['PM25_filtered'],
           a_data_plot_point = f_data_monitors['PM25_filtered'],
           n_lat_min=n_lat_min,
           n_lat_max=n_lat_max,
           n_lon_min=n_lon_min,
           n_lon_max=n_lon_max,
           Colorbar_Label = 'PM$_{2.5}$ [µg/m$^{3}$]',
           Plot_Title = f'Comparing MERRA-2 CNN to Monitors\n(Average {t_start} to {t_end})',
           Colorbar_Limits=[0,50]);

## Comparing Monthly Averages from SatPM and Monitors

To compare the data from the monitors to the SatPM estimate, I first want to average the monitor data to a monthly time scale, so that we can compare the values directly to the monthly averages of the SatPM dataset.

I use `n_minimum_samples_per_month` to specify a minumum number of hourly samples needed to compute a monthly average; this ensures my average will be representative of the whole month, not just a few days or hours.

This code also saves the resulting monthly averaged data to a CSV file, in case I want to look at it later.

In [ ]:
# Specify the minimum number of valid samples for computing a monthly average:
n_minimum_samples_per_month = 24*20

# Create a new data frame for helping to compute the monthly averages:
f_data_monitors_monthly = f_data_monitors.copy().reset_index()
# Add start-of-month time column:
f_data_monitors_monthly['time_month_start'] = [np.datetime64(t_time).astype('datetime64[M]') for t_time in f_data_monitors_monthly['time']]

# Compute monthly averages and standard deviations:
f_data_combined_monthly = f_data_monitors_monthly.groupby(['lat','lon','time_month_start'])[['PM25_filtered']].count().rename(columns={'PM25_filtered':'Count'})
f_data_combined_monthly['Average'] = f_data_monitors_monthly.groupby(['lat','lon','time_month_start'])[['PM25_filtered']].mean()
f_data_combined_monthly['StDev'] = f_data_monitors_monthly.groupby(['lat','lon','time_month_start'])[['PM25_filtered']].std()
# Only keep monthly averages which meet the minimum number of required measurements:
f_data_combined_monthly = f_data_combined_monthly.where(f_data_combined_monthly['Count'] >= n_minimum_samples_per_month)

# Add the monthly SatPM data into this data frame:
f_data_combined_monthly['SatPM'] = np.nan
for i_row in range(len(f_data_combined_monthly)):
  t_month = f_data_combined_monthly.index[i_row][2]
  n_lat = f_data_combined_monthly.index[i_row][0]
  n_lon = f_data_combined_monthly.index[i_row][1]
  f_data_combined_monthly.loc[(n_lat,n_lon,t_month),'SatPM'] = a_data_SatPM.sel(lat=n_lat,lon=n_lon,time=t_month,method='nearest')['PM25_filtered'].values

# Add the monthly MERRA-2 CNN data into this data frame:
f_data_combined_monthly['MERRA2_CNN'] = np.nan
for i_row in range(len(f_data_combined_monthly)):
  t_month = f_data_combined_monthly.index[i_row][2]
  n_lat = f_data_combined_monthly.index[i_row][0]
  n_lon = f_data_combined_monthly.index[i_row][1]
  f_data_combined_monthly.loc[(n_lat,n_lon,t_month),'MERRA2_CNN'] = a_data_MERRA2_CNN.sel(lat=n_lat,lon=n_lon,method='nearest')['PM25_filtered'].sel(time=slice(t_month,t_month+pd.DateOffset(months=1))).mean('time').values

# Save the results into a CSV file for later reference:
f_data_combined_monthly.to_csv(os.path.join(s_folder_Main,'Combined_data_monthly.csv'))

# Look at the resulting data frame:
f_data_combined_monthly

Now I visualize the comparison with a scatterplot and compute some standard performance metrics.

In [ ]:
# Make a comparison scatterplot:
o_ax = sns.scatterplot(data=f_data_combined_monthly.dropna(),x='Average',y='SatPM',hue='time_month_start');
o_ax.plot([0,80],[0,80],'k--');
o_ax.legend(title='Start of Month')
o_ax.set_xlabel('Monitor Monthly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_ylabel('SatPM Monthly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title('Monthly comparison: SatPM v. Ground Monitors');

# Compute and plot performance metrics:
print('Performance Metrics for SatPM')
F_compute_metrics(f_data_combined_monthly,
                  s_time = 'time_month_start',
                  s_value = 'Average',
                  s_estimate = 'SatPM');

In [ ]:
# Make a comparison scatterplot:
o_ax = sns.scatterplot(data=f_data_combined_monthly.dropna(),x='Average',y='MERRA2_CNN',hue='time_month_start');
o_ax.plot([0,80],[0,80],'k--');
o_ax.legend(title='Start of Month')
o_ax.set_xlabel('Monitor Monthly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_ylabel('MERRA-2 CNN Monthly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title('Monthly comparison: MERRA-2 CNN v. Ground Monitors');

# Compute and plot performance metrics:
print('Performance Metrics for MERRA-2 CNN (monthly)')
F_compute_metrics(f_data_combined_monthly,
                  s_time = 'time_month_start',
                  s_value = 'Average',
                  s_estimate = 'MERRA2_CNN');

## Comparing Hourly Averages from MERRA-2 CNN and Monitors

To compare the data from the monitors to the MERRA-2 CNN estimate, I can do a direct hour-to-hour comparison, since they both provide hourly data. However, the spatial resolution is much coarser, with one grid of MERRA-2 usually covering multiple sites. So I first want to aggregate the monitor data to the MERRA-2 grid cells, to allow direct comparability.

I use `n_minimum_samples_per_grid` to specify a minumum number of monitor samples needed to compute an average in a MERRA-2 grid; this ensures my average will be representative of a few monitors in the grid, not just one specific location.

This code also saves the resulting gridded data to a CSV file, in case I want to look at it later.

In [ ]:
# Specify the minimum number of valid samples for computing a grid average:
n_minimum_samples_per_grid = 3

# Create a new data frame for helping to compute the grid averages:
f_data_monitors_gridded = f_data_monitors.copy().reset_index()

# Assign the monitor data to the nearest MERRA-2 grid (note the MERRA-2 grid resolution is 0.5 latitude by 0.625 longitude):
f_data_monitors_gridded['lat_grid'] = round(f_data_monitors_gridded['lat']/0.5)*0.5
f_data_monitors_gridded['lon_grid'] = round(f_data_monitors_gridded['lon']/0.625)*0.625

# Group the data by the latitude and longitude grids and aggregate:
f_data_combined_gridded = f_data_monitors_gridded.groupby(['lat_grid','lon_grid','time'])[['PM25_filtered']].count().rename(columns={'PM25_filtered':'Count'})
f_data_combined_gridded['Average'] = f_data_monitors_gridded.groupby(['lat_grid','lon_grid','time'])[['PM25_filtered']].mean()
f_data_combined_gridded['StDev'] = f_data_monitors_gridded.groupby(['lat_grid','lon_grid','time'])[['PM25_filtered']].std()

# Filter out data with fewer than the required number of samples per grid:
f_data_combined_gridded = f_data_combined_gridded.where(f_data_combined_gridded['Count'] >= n_minimum_samples_per_grid).dropna()

# Add the hourly MERRA-2 CNN data into this data frame (note I also add hour of day and local date columns, to make later steps easier):
f_data_combined_gridded['MERRA2_CNN'] = np.nan
f_data_combined_gridded['local_day'] = np.nan
f_data_combined_gridded['local_hour_of_day'] = np.nan
for i_row in range(len(f_data_combined_gridded)):
  t_time = f_data_combined_gridded.index[i_row][2]
  n_lat = f_data_combined_gridded.index[i_row][0]
  n_lon = f_data_combined_gridded.index[i_row][1]
  f_data_combined_gridded.loc[(n_lat,n_lon,t_time),'MERRA2_CNN'] = a_data_MERRA2_CNN.sel(lat=n_lat,lon=n_lon,time=np.datetime64(t_time.split('+')[0]),method='nearest')['PM25_filtered'].values
  f_data_combined_gridded.loc[(n_lat,n_lon,t_time),'local_day'] = pd.to_datetime(t_time).tz_convert(s_timezome).date()
  f_data_combined_gridded.loc[(n_lat,n_lon,t_time),'local_hour_of_day'] = pd.to_datetime(t_time).tz_convert(s_timezome).hour

# Save the results into a CSV file for later reference:
f_data_combined_gridded.to_csv(os.path.join(s_folder_Main,'Combined_data_gridded.csv'))

# Look at the resulting data frame:
f_data_combined_gridded

Now I visualize the comparison with a scatterplot and compute some standard performance metrics.

In [ ]:
# Make a comparison scatterplot:
o_ax = sns.scatterplot(data=f_data_combined_gridded.dropna(),x='Average',y='MERRA2_CNN',hue='local_hour_of_day')
o_ax.plot([0,200],[0,200],'k--');
o_ax.legend(title='Local Hour of Day');
o_ax.set_xlabel('Monitor Hourly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_ylabel('MERRA-2 CNN Hourly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title('Hourly comparison: MERRA-2 CNN v. Ground Monitors');

# Compute and plot performance metrics:
print('Performance Metrics for MERRA-2 CNN (hourly)')
F_compute_metrics(f_data_combined_gridded,
                  s_time = 'time',
                  s_value = 'Average',
                  s_estimate = 'MERRA2_CNN');

Another way to assess these results is by comparing the diel patterns (i.e., the 24-hour cycle of concentrations) between MERRA-2 CNN and the Monitors.

In [ ]:
# Group the monitor data by local time of day and compute the median:
f_data_combined_gridded_by_timeofday = f_data_combined_gridded.groupby('local_hour_of_day')[['Average']].median()
# Add the count of valid samples:
f_data_combined_gridded_by_timeofday['Count'] = f_data_combined_gridded.groupby('local_hour_of_day')[['Average']].count()
# Add the 25th and 75th percentiles of the hourly averages by time of day, to give a sense of variability:
f_data_combined_gridded_by_timeofday['75th_percentile'] = f_data_combined_gridded.groupby('local_hour_of_day')[['Average']].quantile(0.75)
f_data_combined_gridded_by_timeofday['25th_percentile'] = f_data_combined_gridded.groupby('local_hour_of_day')[['Average']].quantile(0.25)
# Repeat for the MERRA-2 CNN dataset:
f_data_combined_gridded_by_timeofday['MERRA2_CNN'] = f_data_combined_gridded.groupby('local_hour_of_day')[['MERRA2_CNN']].median()
f_data_combined_gridded_by_timeofday['MERRA2_CNN_75th_percentile'] = f_data_combined_gridded.groupby('local_hour_of_day')[['MERRA2_CNN']].quantile(0.75)
f_data_combined_gridded_by_timeofday['MERRA2_CNN_25th_percentile'] = f_data_combined_gridded.groupby('local_hour_of_day')[['MERRA2_CNN']].quantile(0.25)

# Plot the results:
o_fig = plt.figure()
o_ax = o_fig.add_subplot()
o_ax.fill_between(np.arange(0,24),f_data_combined_gridded_by_timeofday['25th_percentile'],f_data_combined_gridded_by_timeofday['75th_percentile'],color='black',alpha=0.25)
o_ax.fill_between(np.arange(0,24),f_data_combined_gridded_by_timeofday['MERRA2_CNN_25th_percentile'],f_data_combined_gridded_by_timeofday['MERRA2_CNN_75th_percentile'],color='red',alpha=0.25)
o_ax.plot(np.arange(0,24),f_data_combined_gridded_by_timeofday['Average'],color='black',label='Monitor Median')
o_ax.plot(np.arange(0,24),f_data_combined_gridded_by_timeofday['MERRA2_CNN'],color='red',label='MERRA-2 CNN Median')
o_ax.legend();
o_ax.set_xlabel('Local Hour of Day');
o_ax.set_xticks([0,3,6,9,12,15,18,21])
o_ax.set_xlim([0,23])
o_ax.set_ylabel('Diel PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title('Diel Cycle comparison: MERRA-2 CNN v. Ground Monitors');

## Comparing Daily Averages from MERRA-2 CNN and Monitors

Let's see if aggregating the data to daily instead of hourly averages will change the comparison and statistics.

In [ ]:
# Aggregate the data according to the local day:
f_data_combined_gridded_daily = f_data_combined_gridded.groupby(['lat_grid','lon_grid','local_day'])[['Average','MERRA2_CNN']].mean()

# Make a comparison scatterplot:
o_ax = sns.scatterplot(data=f_data_combined_gridded_daily.dropna(),x='Average',y='MERRA2_CNN',hue='lat_grid');
o_ax.plot([0,100],[0,100],'k--');
o_ax.legend(title='Latitude');
o_ax.set_xlabel('Monitor Daily Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_ylabel('MERRA-2 CNN Daily Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title('Daily comparison: MERRA-2 CNN v. Ground Monitors');

# Compute and plot performance metrics:
print('Performance Metrics for MERRA-2 CNN (daily)')
F_compute_metrics(f_data_combined_gridded_daily,
                  s_time = 'local_day',
                  s_value = 'Average',
                  s_estimate = 'MERRA2_CNN');

## Example of an analysis using daily average thresholds

It might be of interest to look at data where concentrations are above or below a certain threshold, to compare performance during "clean" versus "polluted" days. Here, I use `n_threshold` to specify a threshold PM<sub>2.5</sub> value for the daily average. Based on whether the daily average is above or below the threshold, I divide the hourly dataset into two parts. I then calculate the statistics for only values where the daily average is below the threshold, for example, to examine the performance on "relatively clean" days.

In [ ]:
# Set the desired threshold for analysis:
n_threshold = 100

# Identify hourly values for which the daily average is above or below the threshold:
f_data_combined_gridded['Daily_Average_Above_Threshold'] = False
for i_row in range(len(f_data_combined_gridded)):
  n_lat = f_data_combined_gridded.index[i_row][0]
  n_lon = f_data_combined_gridded.index[i_row][1]
  t_time = f_data_combined_gridded.index[i_row][2]
  t_day = f_data_combined_gridded.loc[(n_lat,n_lon,t_time),'local_day']
  n_daily_average = f_data_combined_gridded_daily.loc[(n_lat,n_lon,t_day),'Average']
  if n_daily_average > n_threshold:
    f_data_combined_gridded.loc[(n_lat,n_lon,t_time),'Daily_Average_Above_Threshold'] = True

# Plot the results:
o_ax = sns.scatterplot(data=f_data_combined_gridded.dropna(),x='Average',y='MERRA2_CNN',hue='Daily_Average_Above_Threshold')
o_ax.plot([0,200],[0,200],'k--');
o_ax.legend(title='Daily Average Above '+str(n_threshold)+' $\\mu$g/m$^{3}$');
o_ax.set_xlabel('Monitor Hourly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_ylabel('MERRA-2 CNN Hourly Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title('Hourly comparison: MERRA-2 CNN v. Ground Monitors');

# Compute and plot performance metrics, considering only data below threshold:
print('Performance Metrics for MERRA-2 CNN (values below threshold):')
F_compute_metrics(f_data_combined_gridded[f_data_combined_gridded['Daily_Average_Above_Threshold'] == False],
                  s_time = 'time',
                  s_value = 'Average',
                  s_estimate = 'MERRA2_CNN');

## Time Series Plot for a Location

Finally, let's visualize these daily data as a time series for a location. This can be done by restricting our dataset to only the coordinates of one location of interest. In the code below, I've specified some locations of interest in our study region by their corresponding coordinates in the MERRA-2 CNN dataset. A custom list of coordinates of interest could be defined for your study region.

In [ ]:
# Locations:
#   Santiago:                lat -33.5, lon -70.625
#   Vina del Mar:            lat -33.0, lon -71.25
#   Talcahuano y Concepción: lat -37.0, lon -73.125

# This is our location of interest
s_location = 'Talcahuano y Concepción'

# This is a list of locations of interest and their corresponding grid coordinates from the MERRA-2 CNN dataset:
d_location = {'Santiago':(-33.5,-70.625),
              'Viña del Mar':(-33.0,-71.25),
              'Talcahuano y Concepción':(-37.0,-73.125),
              }

# Select only the data for the grid cell corresponding to the location of interest:
f_data_combined_gridded_daily_at_Location = f_data_combined_gridded_daily.loc[d_location[s_location]]

# Plot the time series:
o_fig = plt.figure(figsize=(15,5))
o_ax = o_fig.add_subplot()
o_ax.plot(f_data_combined_gridded_daily_at_Location.index,f_data_combined_gridded_daily_at_Location['Average'],color='black',label='Monitor Average')
o_ax.plot(f_data_combined_gridded_daily_at_Location.index,f_data_combined_gridded_daily_at_Location['MERRA2_CNN'],color='red',label='MERRA-2 CNN Average')
o_ax.legend();
o_ax.set_ylabel('Daily Average PM$_{2.5}$ [$\\mu$g/m$^{3}$]');
o_ax.set_title(f'Location: {s_location}');